## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.


I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

In [1]:
#import everything we need from python, gradio and openai
from dotenv import load_dotenv
from openai import OpenAI 
from pypdf import PdfReader
import gradio as gr


In [2]:
load_dotenv(override=True) # Load environment variables from .env file, overriding existing ones if necessary
openai = OpenAI() # Create an instance of the OpenAI client

In [3]:
reader = PdfReader("./me/Profile.pdf") # Create a PDF reader object for the specified PDF file
linkedln = ""
for page in reader.pages: # Iterate through each page in the PDF
    text = page.extract_text() # Extract text from the current page
    if text:
      linkedln += text # Append the extracted text to the linkedln variable
      

In [4]:
print(linkedln)

   
Contact
4721 SE Wilshire Ter
5806470940 (Mobile)
tango.tew@outlook.com
www.linkedin.com/in/tango-
tew-759437163 (LinkedIn)
Top Skills
Terraform
New Product Rollout
Enterprise Solution Design
Certifications
Python Mega Course
Microsoft Certified: Azure
Administrator Associate
Deep Learning Nanodegree
Build and Deploy Machine Learning
Solutions on Vertex AI Skill Badge
Tango Tew
Senior Technology Consultant @ EY
Edmond, Oklahoma, United States
Summary
Senior Full-Stack and DevOps/Platform Engineer with hands-on
experience across both Microsoft Azure and AWS ecosystems.
I hold a Master of Science in Computer Science & Artificial
Intelligence from SMU, where I specialized in cutting-edge AI
methodologies, including Large Language Model (LLM) applications.
Over the years, I’ve spearheaded innovation and migration initiatives
for cloud-based solutions, leveraging my expertise in full-stack
development, DevOps best practices, and AI-powered frameworks to
deliver transformative results.
Be

In [5]:
with open("./me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Tango" 

In [13]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedln}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}, and be concise and brief. Do not respond with long messages."

In [8]:
system_prompt

"You are acting as Tango. You are answering questions on Tango's website, particularly questions related to Tango's career, background, skills and experience. Your responsibility is to represent Tango for interactions on the website as faithfully as possible. You are given a summary of Tango's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nI am a Senior Technology Consultant and AI Engineer with experience as a DevOps and Platform Engineer, specializing in AWS‑ and Azure‑focused cloud engineering and building custom AI agents that drive measurable productivity gains for engineering teams. Outside of work, he enjoys hiking, praying, reading, watching movies with his family, and occasionally playing basketball, and he holds a Master’s degree in Computer Science with a focus on Artificial Intelligenc

In [14]:
# create a chat function that can be called by gradio, it should take in the user input and return the response from the model
def chat(message, history):
    # create the messages list with the system prompt and the conversation history
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    # call the openai chat completion endpoint with the messages
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    # extract the assistant's reply from the response
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [12]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Missing file: /Users/tango/.cache/huggingface/gradio/frpc/frpc_darwin_arm64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_darwin_arm64
2. Rename the downloaded file to: frpc_darwin_arm64_v0.3
3. Move the file to this location: /Users/tango/.cache/huggingface/gradio/frpc


## Switch into AI workflow using Evaluator optimizer design pattern.
### Key techniques is to call LLM without a framework and have it uses `Structured Output` to return a pass/fail evaluation, and then use that to decide whether to rerun the LLM or not.

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [20]:
# create a pydantic base model for the evaluation structured output that the LLM will use to return its output as code
from pydantic import BaseModel
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [16]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedln}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [18]:
# use gemini as our LLM evaluator, we will call it with the system prompt and the conversation history, and it will return a structured output that we can parse with the Evaluator pydantic model  
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [19]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [25]:
def evaluate(reply, message, history):
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation) # use parse to directly parse the response into our Evaluation pydantic model
    return response.choices[0].message.parsed

In [ ]:
# test if everything works
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [23]:
reply

'No, I do not currently hold a patent. My focus has primarily been on consulting, AI engineering, and cloud solutions. If you have any questions about my work or expertise, feel free to ask!'

In [26]:
# test evaluation
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The agent correctly states that Tango does not hold a patent, as there is no information about patents in the provided context. The response is also professional, engaging, and concise, adhering to all instructions.')

In [27]:
# now lets integrate the whole workflow together where we have Input -> LLM -> Evaluation -> (if fail then back to LLM with feedback)
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n" # for base LLM
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content
    

In [38]:
# lets rebuild our chat function to include evaluation and rerun with feedback if the evaluation fails
def chat(message, history):
    if "patent" in message:
      system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
      system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply = response.choices[0].message.content
    
    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning response")
    else:
        print("Failed evaluation - rerunning with feedback")
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply
    
    

Passed evaluation - returning response


In [39]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - rerunning with feedback
